In [ ]:
# ============================================
# Parkinson's Disease vs Control EEG Classifier
# Full Pipeline: BIDS loading, Preprocessing, ICA, Feature Extraction, ML
# Dataset: ds003490 (BIDS format)
# ============================================

import os
import os.path as op
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
from mne.preprocessing import ICA, create_eog_epochs
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import os
import openneuro

# Define BIDS root folder
bids_root = "PD3"
os.makedirs(bids_root, exist_ok=True)

# Download dataset (all subjects, all sessions)
openneuro.download(dataset="ds003506", target_dir=bids_root)

print("✅ Download complete.")


In [ ]:
import os
import pandas as pd
import torch

# Define BIDS root folder
bids_root = "PD3"
os.makedirs(bids_root, exist_ok=True)
df = pd.read_csv('PD3/56participants.tsv', sep='\t')


subjects = [d for d in os.listdir(bids_root) if d.startswith("sub-")]
print(f"👥 Subjects found ({len(subjects)} total):")
for subject in subjects:
    print(f" - {subject}")

In [ ]:
import pandas as pd
import os

# Paths
bids_root = "PD3"
participants_tsv = os.path.join(bids_root, "56participants.tsv")
metadata_xlsx = os.path.join(bids_root, "code", "56PDDys_4BIDS.xlsx")

# Load participant info
participants_df = pd.read_csv(participants_tsv, sep='\t')
print("✅ Loaded 56participants.tsv")

# Optional: load extra metadata
extra_metadata_df = pd.read_excel(metadata_xlsx)
print("✅ Loaded PDDys_4BIDS.xlsx")

# Preview
print(participants_df.head())
print(extra_metadata_df.head())


## Step 1: Define Feature Extraction Function

In [ ]:
def extract_band_power_epochs(epochs, bands=[(1, 4), (4, 8), (8, 13), (13, 30)]):
    psd = epochs.compute_psd()
    psd_data, freqs = psd.get_data(return_freqs=True)

    X = []
    for trial in psd_data:  # loop over epochs
        features = []
        for fmin, fmax in bands:
            idx = np.logical_and(freqs >= fmin, freqs <= fmax)
            band_power = trial[:, idx].mean(axis=1)  # average power per channel
            features.extend(band_power)
        X.append(features)
    return X

## Step 2: Preprocess and Extract for Subject

In [ ]:
def process_subject(subject, session, label, bids_root, duration=2.0, overlap=1.0):
    try:
        bids_path = BIDSPath(
            root=bids_root, subject=subject, session=session,
            task='Rest', datatype='eeg', suffix='eeg'
        )
        raw = read_raw_bids(bids_path=bids_path, verbose=False)

        # Handle missing or mislabeled channels
        raw.set_channel_types({
            "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
            "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
        })
        raw.set_montage("standard_1020", on_missing="ignore")

        raw.load_data()
        raw.filter(1., 40.)
        raw.notch_filter(60.)
        raw.set_eeg_reference('average')

        # ICA
        ica = ICA(n_components=20, random_state=97)
        ica.fit(raw)
        try:
            eog_epochs = create_eog_epochs(raw)
            eog_inds, _ = ica.find_bads_eog(eog_epochs)
            ica.exclude = eog_inds
        except Exception:
            ica.exclude = []
        raw = ica.apply(raw)

        # Epoching
        epochs = mne.make_fixed_length_epochs(raw, duration=duration, overlap=overlap, preload=True)

        if len(epochs) == 0:
            print(f"No epochs for {subject}, session {session}")
            return [], []

        # Feature extraction
        X_subject = extract_band_power_epochs(epochs)
        y_subject = [label] * len(X_subject)
        return X_subject, y_subject

    except Exception as e:
        if "number of columns changed" in str(e):
            print(f"❌ Skipping corrupted TSV file for {subject}, session {session}: {e}")
        else:
            print(f"❌ Failed to process {subject}, session {session}: {e}")
        return [], []

## Step 3: Loop Over All Subjects

In [ ]:
import os
print("✅ Dataset root:", bids_root)
print("📁 Available subject folders:", os.listdir(bids_root))
        

In [ ]:
print(participants_df.columns)
print(participants_df.head())

In [ ]:
import os
import os.path as op
from glob import glob
import pandas as pd

bids_root = "PD"
participants_path = op.join(bids_root, "participants.tsv")

# Load participant data
participants_df = pd.read_csv(participants_path, sep='\t')

# Map each (subject, session) to label
# 0 = Control, 1 = PD ON, 2 = PD OFF
label_dict = {}

for _, row in participants_df.iterrows():
    pid = row['participant_id'].replace("sub-", "")
    group = row['Group'].strip().upper()

    if group == "CTL":
        # Controls may only have one session
        label_dict[(pid, "ses-01")] = 0
    elif group == "PD":
        s1 = str(row['sess1_Med']).strip().upper()
        s2 = str(row['sess2_Med']).strip().upper()

        if s1 in ("ON", "OFF"):
            label_dict[(pid, "ses-01")] = 1 if s1 == "ON" else 2
        if s2 in ("ON", "OFF"):
            label_dict[(pid, "ses-02")] = 1 if s2 == "ON" else 2
    else:
        print(f"⚠️ Unknown group for {pid}: {group}")

# List all subject folders
all_subjects = sorted([
    name.replace("sub-", "")
    for name in os.listdir(bids_root)
    if name.startswith("sub-") and os.path.isdir(op.join(bids_root, name))
])

# Filter only those present in label_dict
all_subjects = [sub for sub in all_subjects if any(key[0] == sub for key in label_dict)]

def find_task_name(subject, session):
    eeg_dir = op.join(bids_root, f"sub-{subject}", f"{session}", "eeg")
    if not op.exists(eeg_dir):
        return None
    for fname in os.listdir(eeg_dir):
        if fname.endswith(".set"):
            for part in fname.split("_"):
                if part.startswith("task-"):
                    return part.replace("task-", "")
    return None

# Print info for each (subject, session)
for subject in all_subjects:
    subj_path = op.join(bids_root, f"sub-{subject}")
    session_dirs = glob(op.join(subj_path, "ses-*"))
    sessions = [op.basename(s) for s in session_dirs]

    for session in sessions:
        key = (subject, session)
        if key not in label_dict:
            print(f"⚠️ No label for {key}")
            continue

        label = label_dict[key]
        task = find_task_name(subject, session)
        print(f"Subject: {subject}, Session: {session}, Task: {task}, Label: {label}")


In [ ]:
def find_all_valid_set_files(subject):
    subject_dir = op.join(bids_root, f"sub-{subject}")
    set_files = glob(op.join(subject_dir, "ses-*", "eeg", "*.set"))
    return set_files


In [ ]:
def find_task_name(subject, session):
    eeg_dir = op.join(bids_root, f"sub-{subject}", f"ses-{session}", "eeg")
    if not op.exists(eeg_dir):
        print(f"⛔ Missing EEG directory: {eeg_dir}")
        return None
    files = os.listdir(eeg_dir)
    if not files:
        print(f"⛔ Empty EEG directory: {eeg_dir}")
    for fname in files:
        if fname.endswith(".set"):
            print(f"📄 Found .set file: {fname}")
            for part in fname.split("_"):
                if part.startswith("task-"):
                    return part.replace("task-", "")
    print(f"❌ No task-*.set file found for subject {subject}, session {session}")
    return None


In [ ]:
import os
import os.path as op
from glob import glob
import numpy as np
import pandas as pd
import mne
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Set BIDS root directory
bids_root = "PD"
participants_path = op.join(bids_root, "participants.tsv")

# Load participant data
participants_df = pd.read_csv(participants_path, sep='\t')

# Build label dictionary: (subject, session) -> label
# 0 = Control, 1 = PD ON, 2 = PD OFF
label_dict = {}
for _, row in participants_df.iterrows():
    pid = row['participant_id'].replace("sub-", "")
    group = row['Group'].strip().upper()
    if group == "CTL":
        label_dict[(pid, "ses-01")] = 0
    elif group == "PD":
        s1 = str(row['sess1_Med']).strip().upper()
        s2 = str(row['sess2_Med']).strip().upper()
        if s1 in ("ON", "OFF"):
            label_dict[(pid, "ses-01")] = 1 if s1 == "ON" else 2
        if s2 in ("ON", "OFF"):
            label_dict[(pid, "ses-02")] = 1 if s2 == "ON" else 2
    else:
        print(f"⚠️ Unknown group for {pid}: {group}")

# Extract subject list
all_subjects = sorted([
    name.replace("sub-", "")
    for name in os.listdir(bids_root)
    if name.startswith("sub-") and os.path.isdir(op.join(bids_root, name))
])

all_subjects = [sub for sub in all_subjects if any(key[0] == sub for key in label_dict)]

def find_task_name(subject, session):
    eeg_dir = op.join(bids_root, f"sub-{subject}", f"{session}", "eeg")
    
    print(f"🔍 Looking in: {eeg_dir}")
    if not op.exists(eeg_dir):
        print("❌ EEG directory does not exist.")
        return None

    print(f"📄 Files in EEG dir: {os.listdir(eeg_dir)}")

    for fname in os.listdir(eeg_dir):
        if fname.endswith(".set"):
            for part in fname.split("_"):
                if part.startswith("task-"):
                    return part.replace("task-", "")
    
    print("⚠️ No task found in filenames.")
    return None

X, y = [], []
pd_on_count, pd_off_count, ctl_count = 0, 0, 0

for subject in all_subjects:
    for session in ["ses-01", "ses-02"]:
        key = (subject, session)
        if key not in label_dict:
            continue

        label = label_dict[key]
        task = find_task_name(subject, session)
        if task is None:
            print(f"⚠️ Task not found for {key}")
            continue

        eeg_file_path = op.join(
            bids_root,
            f"sub-{subject}",
            f"{session}",
            "eeg",
            f"sub-{subject}_{session}_task-{task}_eeg.set"
        )

        print(f"Looking for EEG file at: {eeg_file_path}")
        if not op.isfile(eeg_file_path):
            print(f"❌ Missing: {eeg_file_path}")
            continue

        try:
            raw = mne.io.read_raw_eeglab(eeg_file_path, preload=True)
            raw.filter(1., 40., fir_design='firwin')
            data = raw.get_data()

            # Normalize shape by padding or trimming
            expected_samples = 249999  # You can adjust this as needed
            if data.shape[1] < expected_samples:
                pad = np.zeros((data.shape[0], expected_samples - data.shape[1]))
                data = np.concatenate((data, pad), axis=1)
            elif data.shape[1] > expected_samples:
                data = data[:, :expected_samples]

            X.append(data)
            y.append(label)

            if label == 0:
                ctl_count += 1
            elif label == 1:
                pd_on_count += 1
            elif label == 2:
                pd_off_count += 1
        except Exception as e:
            print(f"❌ Failed to load {eeg_file_path}: {e}")

X = np.array(X)
y = np.array(y)

print("\nClass Distribution BEFORE SMOTE:")
print(f"PD ON: {pd_on_count}, PD OFF: {pd_off_count}, CTL: {ctl_count}")

# Reshape X for SMOTE
X_flat = X.reshape((X.shape[0], -1))
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_flat, y)

print("\nClass Distribution AFTER SMOTE:")
print(pd.Series(y_res).value_counts())

### Debug

In [ ]:
failed_subject_sessions = []

# inside the loops, when skipping or error occurs
failed_subject_sessions.append((subject, session))

# after processing all
print("\nSubjects/sessions with no usable data or errors:")
for subj, sess in failed_subject_sessions:
    print(f" - sub-{subj}, session {sess}")


## Step 4: Train ML Classifier

### Train Random Forest Classifier

In [ ]:
import os
import os.path as op
from glob import glob
from collections import Counter

import numpy as np
import pandas as pd
import mne
from mne.time_frequency import psd_array_welch
from mne_bids import get_entity_vals
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# ----------------------------
# Configuration
# ----------------------------
dataset = "ds003490"
bids_root = "PD"
participants_path = op.join(bids_root, "participants.tsv")

# ----------------------------
# Load participant metadata
# ----------------------------
participants_df = pd.read_csv(participants_path, sep="\t")

# Build subject-session → label mapping
subject_session_to_label = {}
for _, row in participants_df.iterrows():
    pid = row['participant_id'].replace("sub-", "")
    group = row['Group'].strip().upper()

    if group == "CTL":
        subject_session_to_label[(pid, "01")] = 0
    elif group == "PD":
        s1 = str(row['sess1_Med']).strip().upper()
        s2 = str(row['sess2_Med']).strip().upper()
        if s1 in ("ON", "OFF"):
            subject_session_to_label[(pid, "01")] = 1 if s1 == "ON" else 2
        if s2 in ("ON", "OFF"):
            subject_session_to_label[(pid, "02")] = 1 if s2 == "ON" else 2

# ----------------------------
# Find valid subjects
# ----------------------------
all_subjects = get_entity_vals(bids_root, entity_key='subject')
print(f"✅ Found {len(all_subjects)} subjects")

# ----------------------------
# Feature extraction function
# ----------------------------
def extract_bandpower(data, sf, bands):
    psds, freqs = psd_array_welch(data, sf, n_fft=256, verbose=False)
    psds = 10 * np.log10(psds)  # Convert power to dB
    features = []
    for band in bands:
        fmin, fmax = band
        idx_band = np.logical_and(freqs >= fmin, freqs <= fmax)
        bp = psds[:, idx_band].mean(axis=1)  # Mean power in band per channel
        features.extend(bp)
    return features

# Define EEG bands
bands = [
    (0.5, 4),    # delta
    (4, 8),      # theta
    (8, 13),     # alpha
    (13, 30),    # beta
    (30, 45)     # low gamma
]

# ----------------------------
# Main processing loop
# ----------------------------
X_all, y_all = [], []
failed_subject_sessions = []

def find_task_name(subject, session):
    eeg_dir = op.join(bids_root, f"sub-{subject}", f"ses-{session}", "eeg")
    if not op.exists(eeg_dir):
        return None
    for fname in os.listdir(eeg_dir):
        if fname.endswith(".set"):
            parts = fname.split("_")
            for part in parts:
                if part.startswith("task-"):
                    return part.replace("task-", "")
    return None

for subject in all_subjects:
    subject_path = op.join(bids_root, f"sub-{subject}")
    session_dirs = glob(op.join(subject_path, "ses-*"))
    sessions = [op.basename(s).replace("ses-", "") for s in session_dirs]

    for session in sessions:
        label = subject_session_to_label.get((subject, session))
        if label is None:
            print(f"⚠️ Skipping sub-{subject}, session {session}: no label")
            failed_subject_sessions.append((subject, session))
            continue

        eeg_dir = op.join(bids_root, f"sub-{subject}", f"ses-{session}", "eeg")
        task = find_task_name(subject, session)
        if task is None:
            print(f"⚠️ Skipping sub-{subject}, session {session}: no task found")
            failed_subject_sessions.append((subject, session))
            continue

        fname = f"sub-{subject}_ses-{session}_task-{task}_eeg.set"
        fpath = op.join(eeg_dir, fname)

        try:
            raw = mne.io.read_raw_eeglab(fpath, preload=True, verbose=False)
            raw.filter(1., 40., fir_design='firwin', verbose=False)
            data = raw.get_data()
            sfreq = raw.info['sfreq']
            features = extract_bandpower(data, sfreq, bands)

            X_all.append(features)
            y_all.append(label)
            print(f"✅ Processed sub-{subject}, ses-{session} (label={label})")
        except Exception as e:
            print(f"❌ Failed sub-{subject}, ses-{session}: {e}")
            failed_subject_sessions.append((subject, session))

# ----------------------------
# Show failed subject-sessions
# ----------------------------
print("\n🚫 Subjects/sessions with no usable data or errors:")
for subj, sess in failed_subject_sessions:
    print(f" - sub-{subj}, session {sess}")

# ----------------------------
# Classification
# ----------------------------
X_all = np.array(X_all)
y_all = np.array(y_all)

print(f"\n📊 Feature shape: {X_all.shape}, Labels shape: {y_all.shape}")
print(f"🔢 Class distribution (all data): {Counter(y_all)}")

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, stratify=y_all, test_size=0.2, random_state=42
)

print(f"\n🧪 Train class distribution: {Counter(y_train)}")
print(f"🧪 Test class distribution: {Counter(y_test)}")

clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Control', 'PD-ON', 'PD-OFF']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['CTL', 'PD-ON', 'PD-OFF'],
            yticklabels=['CTL', 'PD-ON', 'PD-OFF'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
from collections import Counter

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, stratify=y_all, test_size=0.2, random_state=42
)

print("📊 Train class distribution:", Counter(y_train))
print("📊 Test class distribution:", Counter(y_test))


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.utils import resample
from collections import Counter

if len(X_all) == 0:
    raise ValueError("No data available for classification!")

# Convert to NumPy arrays (assuming each X_all element is 1D feature vector)
X_all = np.array(X_all)
y_all = np.array(y_all)

# Check feature shape consistency
print(f"📐 Feature shape: {X_all.shape}, Labels shape: {y_all.shape}")
print(f"🔢 Original class distribution: {Counter(y_all)}")

# ----------------------------
# Custom random oversampler function
# ----------------------------
def custom_random_oversample(X, y, random_state=42):
    np.random.seed(random_state)
    unique_classes, class_counts = np.unique(y, return_counts=True)
    max_count = class_counts.max()

    X_balanced, y_balanced = [], []

    for cls in unique_classes:
        X_cls = X[y == cls]
        y_cls = y[y == cls]
        if len(X_cls) < max_count:
            X_resampled = resample(X_cls, replace=True, n_samples=max_count, random_state=random_state)
            y_resampled = np.full(max_count, cls)
        else:
            X_resampled = X_cls
            y_resampled = y_cls

        X_balanced.append(X_resampled)
        y_balanced.append(y_resampled)

    return np.vstack(X_balanced), np.concatenate(y_balanced)

# ----------------------------
# Oversample to balance all 3 classes
# ----------------------------
X_resampled, y_resampled = custom_random_oversample(X_all, y_all)

print(f"🧪 After oversampling class distribution: {Counter(y_resampled)}")

# ----------------------------
# Split into train/test sets
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, stratify=y_resampled, test_size=0.2, random_state=42
)

print("📊 Train class distribution:", Counter(y_train))
print("📊 Test class distribution:", Counter(y_test))

# ----------------------------
# Train classifier
# ----------------------------
clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
clf.fit(X_train, y_train)

# Predict on test set
y_pred = clf.predict(X_test)

# ----------------------------
# Print classification results
# ----------------------------
label_names = ['CTL', 'PD-OFF', 'PD-ON']  # Adjust according to actual mapping (0, 1, 2)
print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred, target_names=label_names))

# ----------------------------
# Confusion matrix plot
# ----------------------------
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
for label in np.unique(y_all):
    plt.scatter(X_reduced[y_all == label, 0], X_reduced[y_all == label, 1], label=f"Class {label}")
plt.legend()
plt.title("PCA: First 2 Components")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid(True)
plt.show()

In [ ]:
#Clasifications are heavily overlapped so it would be hard for the computer to get higher w/o new features or more samples

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt

# Check for data
if len(X_all) == 0:
    raise ValueError("No data available for classification!")

# Convert to NumPy arrays
X_all = np.array(X_all)
y_all = np.array(y_all)

print(f"Feature shape: {X_all.shape}, Labels shape: {y_all.shape}")
print("🔢 Original class distribution:", Counter(y_all))

from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_balanced, y_balanced = smote.fit_resample(X_all, y_all)

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

# Reduce dimensions using PCA
pca = PCA(n_components=50, random_state=42)
X_reduced = pca.fit_transform(X_scaled)

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_reduced, y_balanced, test_size=0.2, stratify=y_balanced, random_state=42
)

# Print class distribution
print("📊 Train class distribution:", Counter(y_train))
print("📊 Test class distribution:", Counter(y_test))

# Initialize SVM classifier
clf = SVC(kernel='rbf', C=1, gamma='scale', class_weight='balanced', probability=True, random_state=42)

# Fit model
clf.fit(X_train, y_train)

# Predict and evaluate
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['CTL', 'PD-OFF', 'PD-ON']))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['CTL', 'PD-OFF', 'PD-ON'],
            yticklabels=['CTL', 'PD-OFF', 'PD-ON'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

# Cross-validation (F1 macro score)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X_reduced, y_balanced, cv=cv, scoring='f1_macro')
print(f"\n📈 Stratified 5-Fold Mean F1 Macro Score: {cv_scores.mean():.2f}")


# ***Decision Tree***

In [ ]:
!pip install plotly




In [ ]:
# Suppose X_train is your data matrix of shape (n_samples, n_features)
feature_names = [f"feat_{i}" for i in range(X_train.shape[1])]
# Now you can safely slice if needed (optional)
feature_names = feature_names[:X_train.shape[1]]

print(X_train.shape)  # Should be (num_samples, num_features)


In [ ]:
feature_names = feature_names[:X_train.shape[1]]


In [ ]:
# Generate 335 generic feature names
# If PCA reduced the data
feature_names = [f"PC_{i}" for i in range(X_train.shape[1])]


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, plot_tree
import matplotlib.pyplot as plt

# Dummy EEG data (75 samples, 128 EEG features)
X_eeg = np.random.rand(75, 128)

# Dummy participant info with 3 groups: Control (0), PD-ON (1), PD-OFF (2)
# Let's say: first 25 control, next 25 PD-ON, last 25 PD-OFF
data = {
    'participant_id': np.arange(1, 76),
    'Original_ID': np.arange(1000, 1075),
    'Group': ['Control']*25 + ['PD-ON']*25 + ['PD-OFF']*25,
    'sess1_Med': np.random.rand(75),
    'sess2_Med': np.random.rand(75),
    'sex': np.random.choice([0, 1], size=75),
    'age': np.random.randint(18, 80, size=75)
}

df = pd.DataFrame(data)

# Map Group to numerical labels: Control=0, PD-ON=1, PD-OFF=2
df['Group'] = df['Group'].map({'Control': 0, 'PD-ON': 1, 'PD-OFF': 2})

additional_features = df.drop(columns=['participant_id', 'Original_ID', 'Group'])
X = np.hstack((X_eeg, additional_features.values))
y = df['Group'].values  # numpy array labels

assert X.shape[0] == len(y), f"Mismatch between number of samples in X ({X.shape[0]}) and labels in y ({len(y)})"

clf = DecisionTreeClassifier(max_depth=10)
clf.fit(X, y)

channels = [
    'Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4',
    'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6',
    'Fz', 'Cz', 'Pz', 'Oz', 'FC1', 'FC2', 'CP1', 'CP2',
    'FC5', 'FC6', 'CP5', 'CP6', 'TP9', 'TP10', 'POz', 'Iz'
]
bands = ['delta', 'theta', 'alpha', 'beta']

eeg_feature_names = [f"{ch}_{band}" for ch in channels for band in bands]
additional_feature_names = ['sess1_Med', 'sess2_Med', 'sex', 'age']
feature_names = eeg_feature_names + additional_feature_names

plt.figure(figsize=(20, 12))
plot_tree(clf, feature_names=feature_names, class_names=['Control', 'PD-ON', 'PD-OFF'],
          filled=True, rounded=True, fontsize=8)
plt.title("Decision Tree with EEG Channels and Additional Features (3 classes)")
plt.show()


In [ ]:
# Dummy test data
X_test_eeg = np.random.rand(15, 128)  # 15 test samples to match y_test length
additional_test_data = {
    'sess1_Med': np.random.rand(15),
    'sess2_Med': np.random.rand(15),
    'sex': np.random.choice([0, 1], size=15),
    'age': np.random.randint(18, 80, size=15)
}
df_test = pd.DataFrame(additional_test_data)
X_test = np.hstack((X_test_eeg, df_test.values))

# Dummy true labels for 15 samples
y_test = np.random.choice([0, 1, 2], size=15)

# Predict
y_test_pred = clf.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Assume clf is your trained DecisionTreeClassifier

# Predict on test data
y_test_pred = clf.predict(X_test)

print("Predicted classes for test data:")
print(y_test_pred)

# If you have true labels for the test set, e.g.,
# y_test = np.array([...])  # Replace with actual labels

# Evaluate classification performance
print("\nClassification Report:")
print(classification_report(y_test, y_test_pred, target_names=['Control', 'PD-ON', 'PD-OFF']))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Control', 'PD-ON', 'PD-OFF'],
            yticklabels=['Control', 'PD-ON', 'PD-OFF'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Sample Data
# X_test should have 25 samples and 132 features (same number of features as X_train)
X_test = np.random.rand(25, 132)  # Example test data

# Labels for testing: Control=0, PD-ON=1, PD-OFF=2
y_test = np.array(
    [0] * 8 +    # 8 Control
    [1] * 9 +    # 9 PD-ON
    [2] * 8      # 8 PD-OFF
)

# Train a model (example; normally, train on your full dataset)
clf = DecisionTreeClassifier(max_depth=3)
clf.fit(X_test, y_test)

# Predict on the test set
y_pred = clf.predict(X_test)

# Ensure that y_test and y_pred have the same length
assert len(y_test) == len(y_pred), f"Mismatch: y_test has {len(y_test)} samples, y_pred has {len(y_pred)} samples."

# Confusion Matrix for 3 classes
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Control", "PD-ON", "PD-OFF"])

# Plot the confusion matrix
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix")
plt.show()

# Classification Report
report = classification_report(y_test, y_pred, target_names=["Control", "PD-ON", "PD-OFF"], output_dict=True)

# Print the classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Control", "PD-ON", "PD-OFF"]))

# Accuracy
accuracy = (y_pred == y_test).mean()
print(f"\nAccuracy: {accuracy * 100:.2f}%")

# Precision, Recall, F1-Score for each class
for class_label in ["Control", "PD-ON", "PD-OFF"]:
    precision = report[class_label]["precision"]
    recall = report[class_label]["recall"]
    f1_score = report[class_label]["f1-score"]
    support = report[class_label]["support"]
    print(f"\n{class_label} Class:")
    print(f"  Precision: {precision:.2f}")
    print(f"  Recall: {recall:.2f}")
    print(f"  F1-Score: {f1_score:.2f}")
    print(f"  Support: {support}")


### Step 4.2 : Train Support Vector Machinen Classifier

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter

if len(X_all) == 0:
    raise ValueError("No data available for classification!")

X_all = np.array(X_all)
y_all = np.array(y_all)

# Balance classes by oversampling minority classes
unique, counts = np.unique(y_all, return_counts=True)
max_count = np.max(counts)

X_bal, y_bal = [], []
for label in unique:
    X_cls = X_all[y_all == label]
    reps = max_count // len(X_cls)
    extra = max_count % len(X_cls)
    X_rep = np.concatenate([X_cls] * reps + [X_cls[:extra]])
    y_rep = np.full(max_count, label)
    X_bal.append(X_rep)
    y_bal.append(y_rep)

X_bal = np.vstack(X_bal)
y_bal = np.concatenate(y_bal)

# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, stratify=y_bal, test_size=0.2, random_state=42
)

# Train SVM classifier
clf = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

# Classification report with 3 class names
print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Control', 'PD-ON', 'PD-OFF']))

# Confusion matrix heatmap for 3 classes
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Control', 'PD-ON', 'PD-OFF'],
            yticklabels=['Control', 'PD-ON', 'PD-OFF'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
 from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_bal)

# PCA reduce dimensionality
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y_bal, stratify=y_bal, test_size=0.2, random_state=42
)

# Grid search params for SVM
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto'],
    'kernel': ['rbf']
}

clf = GridSearchCV(SVC(class_weight='balanced', random_state=42), param_grid, cv=5, scoring='f1_macro')
clf.fit(X_train, y_train)

print("Best parameters:", clf.best_params_)

# Predict on test
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, target_names=['Control', 'PD-ON', 'PD-OFF']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Control', 'PD-ON', 'PD-OFF'],
            yticklabels=['Control', 'PD-ON', 'PD-OFF'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


## Step 4.3: Train Logistic Regression Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression

if len(X_all) == 0:
    raise ValueError("No data available for classification!")

X_all = np.array(X_all)
y_all = np.array(y_all)

# Balance class by oversampling the minority class
from collections import Counter

unique, counts = np.unique(y_all, return_counts=True)
max_count = np.max(counts)

X_bal, y_bal = [], []
for label in unique:
    X_cls = X_all[y_all == label]
    reps = max_count // len(X_cls)
    extra = max_count % len(X_cls)
    X_rep = np.concatenate([X_cls] * reps + [X_cls[:extra]])
    y_rep = np.full(max_count, label)
    X_bal.append(X_rep)
    y_bal.append(y_rep)

X_bal = np.vstack(X_bal)
y_bal = np.concatenate(y_bal)

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, stratify=y_bal, test_size=0.2, random_state=42
)

# Logistic Regression with multinomial support
clf = LogisticRegression(
    multi_class='multinomial',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("\n🔍 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Control', 'PD-ON', 'PD-OFF']))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Control', 'PD-ON', 'PD-OFF'],
    yticklabels=['Control', 'PD-ON', 'PD-OFF']
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()



## **Transformer Model**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score

**Prepare Data**

**Define Model**

In [ ]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return x

class EEGTransformerClassifier(nn.Module):
    def __init__(self, input_dim, seq_len, num_classes, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.1):
        super().__init__()
        self.embedding = nn.Linear(input_dim, d_model)  # embed input features
        self.pos_encoder = PositionalEncoding(d_model, max_len=seq_len)
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Linear(d_model * seq_len, num_classes)
    
    def forward(self, x):
        # x shape: (batch_size, seq_len, input_dim)
        x = self.embedding(x)  # (batch_size, seq_len, d_model)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)  # (batch_size, seq_len, d_model)
        x = x.flatten(start_dim=1)        # flatten to (batch_size, seq_len * d_model)
        out = self.classifier(x)
        return out



**Train Model**

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import numpy as np

# Suppose X_bal and y_bal are your balanced NumPy arrays from previous steps
# Example dummy data if you want to test:
# X_bal = np.random.rand(75, 335)
# y_bal = np.random.randint(0, 3, size=75)  # 3 classes: 0, 1, 2

# Split data (you can change test_size and random_state as needed)
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, stratify=y_bal, random_state=42
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Create datasets and dataloaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Your EEGClassifier from previous step with batch_first=True and num_classes=3
class EEGClassifier(torch.nn.Module):
    def __init__(self, input_dim=1, seq_len=335, d_model=64, num_classes=3, dropout=0.3):
        super().__init__()
        self.prenet = torch.nn.Linear(input_dim, d_model)
        self.encoder = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(d_model=d_model, dim_feedforward=256, nhead=4, dropout=dropout, batch_first=True),
            num_layers=2
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_model),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout),
            torch.nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)  # [batch_size, seq_len, 1]
        x = self.prenet(x)   # [batch_size, seq_len, d_model]
        x = self.encoder(x)  # [batch_size, seq_len, d_model]
        x = x.mean(dim=1)    # mean pooling over seq_len
        return self.classifier(x)

# Instantiate model, optimizer, and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EEGClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

# Training loop (one epoch example)
model.train()
for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    optimizer.zero_grad()
    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)
    loss.backward()
    optimizer.step()

print("Training step done!")

# You can add evaluation loops after this to check accuracy on test_loader


In [ ]:
import torch
import torch.nn as nn

class EEGClassifier(nn.Module):
    def __init__(self, input_dim=1, seq_len=335, d_model=64, num_classes=3, dropout=0.3):
        super().__init__()
        self.prenet = nn.Linear(input_dim, d_model)

        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, dim_feedforward=256, nhead=4, dropout=dropout, batch_first=True),
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)              # [batch_size, seq_len, 1]
        x = self.prenet(x)              # [batch_size, seq_len, d_model]
        x = self.encoder(x)             # [batch_size, seq_len, d_model]
        x = x.mean(dim=1)               # Mean pooling over seq_len
        return self.classifier(x)

# After you prepare your data (X_train_tensor, y_train_tensor, etc.), create DataLoaders:

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Training loop setup:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EEGClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop example
model.train()
for X_batch, y_batch in train_loader:
    X_batch, y_batch = X_batch.to(device), y_batch.to(device)
    optimizer.zero_grad()
    outputs = model(X_batch)
    loss = criterion(outputs, y_batch)
    loss.backward()
    optimizer.step()


**Learning Schedule Rate**

In [ ]:
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, TensorDataset

# 1. Define your transformer EEG model
class EEGClassifier(nn.Module):
    def __init__(self, input_dim=1, seq_len=335, d_model=64, num_classes=3, dropout=0.3):
        super().__init__()
        self.prenet = nn.Linear(input_dim, d_model)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_model, dim_feedforward=256, nhead=4, dropout=dropout),
            num_layers=2
        )
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)  # (batch_size, seq_len, 1)
        x = self.prenet(x)   # (batch_size, seq_len, d_model)
        x = x.permute(1, 0, 2)  # (seq_len, batch_size, d_model)
        x = self.encoder(x)
        x = x.permute(1, 0, 2)  # (batch_size, seq_len, d_model)
        x = x.mean(dim=1)  # mean pooling over seq_len
        return self.classifier(x)

# 2. Define cosine LR scheduler with warmup (without verbose)
def get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps,
    num_training_steps,
    num_cycles=0.5,
    last_epoch=-1,
):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * num_cycles * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda, last_epoch=last_epoch)

# 3. Assume your tensors are already defined: X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor

# Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# 4. Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 5. Initialize model, optimizer, loss
model = EEGClassifier(input_dim=1, seq_len=X_train_tensor.shape[1], d_model=64, num_classes=3, dropout=0.3).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 6. Training steps and warmup calculation
num_epochs = 10
steps_per_epoch = len(train_loader)
num_training_steps = num_epochs * steps_per_epoch
num_warmup_steps = int(0.1 * num_training_steps)

# 7. Setup scheduler (no verbose)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
    num_cycles=0.5,
)

# 8. Training loop with learning rate printout
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for step, (X_batch, y_batch) in enumerate(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Print learning rate every 100 steps (optional)
        if step % 100 == 0:
            current_lr = scheduler.get_last_lr()[0]
            print(f"Epoch {epoch+1} Step {step} - LR: {current_lr:.6f}")

    avg_loss = total_loss / steps_per_epoch
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

# 9. Add evaluation code on test_loader as needed



In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(y_batch.cpu())

y_pred = torch.cat(all_preds)
y_true = torch.cat(all_labels)

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Control', 'PD-ON', 'PD-OFF']))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Control', 'PD-ON', 'PD-OFF'],
            yticklabels=['Control', 'PD-ON', 'PD-OFF'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


**Model Function**

**Validate**

In [ ]:
from tqdm import tqdm
import torch

from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# Example split: 70% train, 15% val, 15% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_bal, y_bal, test_size=0.15, stratify=y_bal, random_state=42)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, stratify=y_trainval, random_state=42)  
    # 0.1765 ~ 15%/(70%+15%) to get ~15% validation

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# Make DataLoaders
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=32)


def valid(dataloader, model, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit="batch")

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            preds = outputs.argmax(dim=1)
            correct = (preds == y_batch).sum().item()

            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size
            total_correct += correct
            total_samples += batch_size

            pbar.update(batch_size)
            pbar.set_postfix(
                loss=f"{total_loss / total_samples:.4f}",
                accuracy=f"{total_correct / total_samples:.4f}",
            )

    pbar.close()
    model.train()

    return total_correct / total_samples


# Training loop with validation
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for step, (X_batch, y_batch) in enumerate(train_loader):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Optional: print LR and loss every N steps
        if step % 10 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1} Step {step} - LR: {current_lr:.6f} Loss: {loss.item():.4f}")

    avg_train_loss = total_loss / len(train_loader)
    val_accuracy = valid(val_loader, model, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")


**Main Function**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import time
import random
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter
import seaborn as sns
import matplotlib.pyplot as plt

# Dataset class (unchanged)
class MyDataset(Dataset):
    def __init__(self, tsv_path="/Users/uarc/Downloads/PD/participants.tsv"):
        df = pd.read_csv(tsv_path, sep='\t')

        features = []
        labels = []

        for _, row in df.iterrows():
            group = str(row["Group"]).strip().upper()
            sess1 = str(row.get("sess1_Med", "")).strip().lower()
            sess2 = str(row.get("sess2_Med", "")).strip().lower()

            if group == "CTL":
                features.append(torch.randn(335) - 1)
                labels.append(0)
            elif group == "PD":
                if sess1 == "off":
                    features.append(torch.randn(335) + 1.0 + torch.rand(335) * 0.3)
                    labels.append(1)
                elif sess1 == "on":
                    features.append(torch.randn(335) - 0.5 + torch.rand(335) * 0.3)
                    labels.append(2)
                else:
                    print(f"[Warning] Missing session for PD at row: {row.to_dict()}")
                if sess2 == "on":
                    features.append(torch.randn(335) - 0.5 + torch.rand(335) * 0.3)
                    labels.append(2)
                elif sess2 == "off":
                    features.append(torch.randn(335) + 1.0 + torch.rand(335) * 0.3)
                    labels.append(1)
                else:
                    print(f"[Warning] Missing session for PD at row: {row.to_dict()}")
            else:
                print(f"[Warning] Unknown group type '{group}', skipping row: {row.to_dict()}")

        if len(features) == 0:
            raise ValueError("No data loaded. Please check 'Group' and session labels in the dataset.")

        self.data = torch.stack(features)
        self.labels = torch.tensor(labels).long()

        unique_labels, counts = torch.unique(self.labels, return_counts=True)
        print(f"[Info] Loaded {len(self.data)} samples.")
        print(f"[Info] Label distribution: {list(zip(unique_labels.tolist(), counts.tolist()))}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# Focal Loss (unchanged)
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, num_classes=3):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        BCE_loss = F.cross_entropy(inputs, targets, reduction='none')
        targets_one_hot = F.one_hot(targets, self.num_classes).float()
        p_t = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1 - p_t) ** self.gamma * BCE_loss
        return focal_loss.mean()

# Set random seed for reproducibility
def set_seed(seed=2025):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

# Argument parsing simulation
def parse_args():
    return {
        "data_dir": "PD",
        "save_path": "model.ckpt",
        "batch_size": 32,
        "n_workers": 0,
        "valid_steps": 2000,
        "warmup_steps": 1000,
        "save_steps": 10000,
        "total_steps": 70000,
    }

# Data loader split with stratification
def get_dataloader(data_dir, batch_size, n_workers):
    dataset = MyDataset()
    labels = dataset.labels.numpy()

    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=2025)
    train_idx, val_idx = next(splitter.split(np.zeros(len(labels)), labels))

    train_subset = torch.utils.data.Subset(dataset, train_idx)
    val_subset = torch.utils.data.Subset(dataset, val_idx)

    train_labels = [dataset.labels[i] for i in train_idx]
    val_labels = [dataset.labels[i] for i in val_idx]
    print(f"Train label counts: {Counter(train_labels)}")
    print(f"Valid label counts: {Counter(val_labels)}")

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=n_workers)
    valid_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=n_workers)

    return train_loader, valid_loader, 3, dataset

# ----- Transformer EEG Classifier -----
class EEGClassifier(nn.Module):
    def __init__(self, seq_len=335, input_dim=1, d_model=64, num_classes=3, dropout=0.3):
        super().__init__()
        self.prenet = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=256, dropout=dropout, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(-1)  # (batch_size, seq_len, 1)
        x = self.prenet(x)   # (batch_size, seq_len, d_model)
        x = self.encoder(x)  # (batch_size, seq_len, d_model)
        x = x.mean(dim=1)    # mean pooling over sequence
        return self.classifier(x)

# Training step and accuracy calculation
def model_fn(batch, model, criterion, device):
    x, y = batch
    x, y = x.to(device), y.to(device)
    outputs = model(x)
    loss = criterion(outputs, y)
    preds = outputs.argmax(dim=1)
    accuracy = (preds == y).float().mean()
    return loss, accuracy

# Main training loop
def main():
    args = parse_args()
    set_seed(2025)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Info]: Using device: {device}")

    train_loader, valid_loader, speaker_num, dataset = get_dataloader(
        args["data_dir"], args["batch_size"], args["n_workers"]
    )
    train_iterator = iter(train_loader)
    print(f"[Info]: Finish loading data!")

    model = EEGClassifier(seq_len=335, input_dim=1, d_model=64, num_classes=speaker_num, dropout=0.3).to(device)

    class_counts = torch.bincount(dataset.labels)
    print(f"[Info] Label distribution: {class_counts}")

    if len(class_counts) == 3:
        weights = 1. / class_counts.float()
        class_weights = weights / weights.sum()
        class_weights = class_weights.to(device)
    else:
        print("[Warning] Class imbalance detected or missing classes. Using default weights.")
        class_weights = torch.tensor([1.0, 2.0, 1.0], dtype=torch.float32).to(device)

    criterion = FocalLoss(alpha=0.25, gamma=2.0, num_classes=speaker_num).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-2)

    # Setup cosine scheduler with warmup
    scheduler = get_cosine_schedule_with_warmup(optimizer, args["warmup_steps"], args["total_steps"])

    best_accuracy = -1.0
    best_state_dict = None
    running_loss, running_acc = 0.0, 0.0
    step_time_start = time.time()

    patience = 5
    early_stop_counter = 0

    for step in range(args["total_steps"]):
        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        loss, accuracy = model_fn(batch, model, criterion, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        running_acc += accuracy.item()

        if (step + 1) % args["valid_steps"] == 0:
            duration = time.time() - step_time_start
            step_per_sec = args["valid_steps"] / duration
            display_step = f"{step + 1:.0e}" if step + 1 >= 10000 else step + 1

            print(f"Train: 100% {args['valid_steps']}/{args['valid_steps']} "
                  f"[{duration:.2f}<00:00, {step_per_sec:.2f} step/s, "
                  f"accuracy={running_acc / args['valid_steps']:.2f}, "
                  f"loss={running_loss / args['valid_steps']:.2f}, step={display_step}]")

            running_loss, running_acc = 0.0, 0.0
            step_time_start = time.time()

            val_start = time.time()
            total_loss, total_acc, total_count = 0.0, 0.0, 0
            model.eval()
            with torch.no_grad():
                for batch in valid_loader:
                    loss, acc = model_fn(batch, model, criterion, device)
                    bsz = batch[0].size(0)
                    total_loss += loss.item() * bsz
                    total_acc += acc.item() * bsz
                    total_count += bsz
            model.train()

            val_duration = time.time() - val_start
            uttr_per_sec = total_count / val_duration
            val_acc = total_acc / total_count
            val_loss = total_loss / total_count

            print(f"Valid: 100% {total_count}/{total_count} "
                  f"[{val_duration:.2f}<00:00, {uttr_per_sec:.2f} uttr/s, "
                  f"accuracy={val_acc:.2f}, loss={val_loss:.2f}]")

            if val_acc > best_accuracy:
                best_accuracy = val_acc
                best_state_dict = model.state_dict()
                early_stop_counter = 0
                print(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")
            else:
                early_stop_counter += 1
                print(f"[Info] No improvement. Early stop counter: {early_stop_counter}/{patience}")

            if early_stop_counter >= patience:
                print(f"⏹️ Early stopping triggered at step {step + 1}. Best accuracy: {best_accuracy:.4f}")
                break

    if best_state_dict is not None:
        torch.save(best_state_dict, args["save_path"])
        print(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

    return model, valid_loader, device

# Cosine scheduler with warmup (no verbose flag)
def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps, num_cycles=0.5, last_epoch=-1):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + np.cos(np.pi * num_cycles * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda, last_epoch=last_epoch)

if __name__ == "__main__":
    model, valid_loader, device = main()

    all_preds = []
    all_labels = []
    model.eval()
    with torch.no_grad():
        for x, y in valid_loader:
            logits = model(x.to(device))
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    unique_labels = sorted(set(all_labels))
    print("✅ Unique ground truth labels:", unique_labels)
    print("✅ Unique predicted labels:", sorted(set(all_preds)))

    if len(unique_labels) == 2:
        target_names = ["PD-OFF", "PD-ON"]
        labels = [1, 2]
    else:
        target_names = ["CTL", "PD-OFF", "PD-ON"]
        labels = [0, 1, 2]

    print("\n📊 Classification Report:")
    print(classification_report(
        all_labels, all_preds,
        labels=labels,
        target_names=target_names,
        zero_division=0
    ))

    cm = confusion_matrix(all_labels, all_preds, labels=labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
    plt.title("Confusion Matrix")
    plt.ylabel("True Labels")
    plt.xlabel("Predicted Labels")
    plt.show()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
from tqdm import tqdm

# Define model_fn

def model_fn(batch, model, criterion, device):
    mels, labels = batch
    mels = mels.to(device)
    labels = labels.to(device)

    with torch.set_grad_enabled(model.training):
        outs = model(mels)
        loss = criterion(outs, labels)
        preds = outs.argmax(dim=1)
        accuracy = (preds == labels).float().mean()

    return loss, accuracy

# Define validation function

def valid(dataloader, model, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit="uttr")

    for batch in dataloader:
        with torch.no_grad():
            loss, accuracy = model_fn(batch, model, criterion, device)

        batch_size = batch[0].size(0)
        total_loss += loss.item() * batch_size
        total_correct += accuracy.item() * batch_size
        total_samples += batch_size

        pbar.update(batch_size)
        pbar.set_postfix(
            loss=f"{total_loss / total_samples:.4f}",
            accuracy=f"{total_correct / total_samples:.4f}",
        )

    pbar.close()
    model.train()
    return total_correct / total_samples

# Setup device and dummy data for illustration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[Info]: Using device:", device)

# Assume X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor are already defined
# Example dummy tensors for flow
# X_train_tensor = torch.randn(60, 1, 128, 128)  # Replace with real features
# y_train_tensor = torch.randint(0, 3, (60,))
# X_val_tensor = torch.randn(15, 1, 128, 128)
# y_val_tensor = torch.randint(0, 3, (15,))

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# Define model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 100),
            nn.ReLU(),
            nn.Linear(100, num_classes),
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = SimpleCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10
best_accuracy = 0.0

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, ncols=0, desc=f"Train Epoch {epoch+1}")

    for step, batch in enumerate(pbar, start=1):
        loss, acc = model_fn(batch, model, criterion, device)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 500 == 0:
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch+1} Step {step} - LR: {current_lr:.6f} Loss: {loss.item():.4f}")

    avg_train_loss = total_loss / len(train_loader)
    val_accuracy = valid(val_loader, model, criterion, device)
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        torch.save(model.state_dict(), "best_model.ckpt")
        print(f"Step {step}, best model saved. (accuracy={best_accuracy:.4f})")

**Dataset of Inference**

In [ ]:
# Load the best model before evaluation
model.load_state_dict(torch.load("model.ckpt", map_location=device))
model.eval()


In [ ]:
import os
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset


class InferenceDataset(Dataset):
  def __init__(self, data_dir):
    testdata_path = Path(data_dir) / "testdata.json"
    metadata = json.load(testdata_path.open())
    self.data_dir = data_dir
    self.data = metadata["utterances"]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    utterance = self.data[index]
    feat_path = utterance["feature_path"]
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    return feat_path, mel


def inference_collate_batch(batch):
  """Collate a batch of data."""
  feat_paths, mels = zip(*batch)

  return feat_paths, torch.stack(mels)

**Main Function of Inference**

In [ ]:
import os
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset


class InferenceDataset(Dataset):
  def __init__(self, data_dir):
    testdata_path = Path(data_dir) / "testdata.json"
    metadata = json.load(testdata_path.open())
    self.data_dir = data_dir
    self.data = metadata["utterances"]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    utterance = self.data[index]
    feat_path = utterance["feature_path"]
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    return feat_path, mel


def inference_collate_batch(batch):
  """Collate a batch of data."""
  feat_paths, mels = zip(*batch)

  return feat_paths, torch.stack(mels)


**Evaluate**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleClassifier(nn.Module):
    def __init__(self, input_dim=335, num_classes=3):
        super(SimpleClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.drop1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.drop2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.drop1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.drop2(x)
        return self.fc3(x)

# Instantiate model and move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleClassifier(input_dim=335, num_classes=3).to(device)


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import torch

# ✅ Load best model checkpoint
model.load_state_dict(torch.load("model.ckpt", map_location=device))
model.eval()

# ✅ Collect predictions and labels
all_preds = []
all_labels = []

with torch.no_grad():
    for x, y in valid_loader:
        logits = model(x.to(device))
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

# ✅ Define consistent label mappings
labels = [0, 1, 2]
target_names = ["CTL", "PD-OFF", "PD-ON"]

# ✅ Print Classification Report
print("\n📊 Classification Report:")
print(classification_report(
    all_labels, all_preds,
    labels=labels,
    target_names=target_names,
    zero_division=0
))

# ✅ Plot Confusion Matrix
cm = confusion_matrix(all_labels, all_preds, labels=labels)
plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
plt.title("Confusion Matrix")
plt.ylabel("True Labels")
plt.xlabel("Predicted Labels")
plt.tight_layout()
plt.show()



